In [ ]:
import os
import rasterio
from rasterio.merge import merge

# ==============================================================================
# LOAD DATA
# ==============================================================================
# 1. Load Data
from google.colab import drive

# Check if drive is already mounted and unmount if it is, to prevent 'Mountpoint must not already contain files' error
if os.path.exists('/content/drive/My Drive'):
    print("Google Drive is already mounted. Unmounting first...")
    drive.flush_and_unmount()

drive.mount('/content/drive')

# ==========================================================
# 1. PENGATURAN DIREKTORI
# ==========================================================
BASE_DIR = '/content/drive/MyDrive/Data Skripsi Pemetaan'
OUTPUT_DIR = os.path.join(BASE_DIR, 'Hasil_Prediksi_Model_Skenario 9') # Nama folder diperbarui
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 6 Periode yang akan kita gabungkan
periodes = ['02', '04', '06', '08', '10', '12']

# WAJIB: Definisi colormap yang SUDAH DIPERBAIKI (6=Bera, 7=Bukan Amatan)
warna_fase = {
    0: (0, 0, 0, 0),         # 0. Transparan (Luar Area Sawah/Kabupaten)
    1: (144, 238, 144, 255), # 1. Vegetatif Awal (Hijau Muda)
    2: (34, 139, 34, 255),   # 2. Vegetatif Akhir (Hijau Tua)
    3: (255, 255, 0, 255),   # 3. Generatif (Kuning)
    4: (255, 165, 0, 255),   # 4. Panen (Orange)
    5: (139, 69, 19, 255),   # 5. Persiapan Lahan (Coklat)
    6: (211, 211, 211, 255), # 6. Bera (Abu-abu)
}

print(f"{'='*60}")
print("🌍 MEMULAI PENGGABUNGAN PETA (HANYA DARI FOLDER BARU)")
print(f"{'='*60}")

# ==========================================================
# 2. PROSES PENCARIAN & PENGGABUNGAN PER PERIODE
# ==========================================================
for per in periodes:
    print(f"\n⏳ Memproses Peta Periode {per}...")

    file_list = []

    # "Radar Otomatis": Kelilingi semua folder di dalam Data Skripsi Pemetaan
    for root, dirs, files in os.walk(BASE_DIR):

        # --- KUNCI UTAMA: HANYA MASUK KE FOLDER BARU ---
        # Jika nama folder saat ini bukan 'Hasil_Prediksi_Model_Baru_6Label', langsung lewati!
        if 'Hasil_Prediksi_Model_Skenario 9' not in root:
            continue

        for file in files:
            # Ubah nama file ke huruf kecil semua agar pencarian kebal typo
            nama_file_kecil = file.lower()

            # RADAR AGRESIF: Cari file yang ada kata "pred", ada "02.tif" dll
            if "pred" in nama_file_kecil and f"_{per}.tif" in nama_file_kecil:
                file_list.append(os.path.join(root, file))

    # Cek apakah ada file yang ditemukan
    if not file_list:
        print(f"   ⚠️ Melewati Periode {per}: Tidak ada file prediksi yang ditemukan.")
        continue

    print(f"   🔍 Ditemukan {len(file_list)} kecamatan. Memulai penjahitan (merge)...")

    try:
        # Buka semua file TIF yang ditemukan ke dalam memori
        src_files_to_mosaic = []
        for fp in file_list:
            src = rasterio.open(fp)
            src_files_to_mosaic.append(src)

        # PROSES INTI: Jahit semua peta menjadi satu kanvas besar!
        mosaic, out_trans = merge(src_files_to_mosaic)

        # Salin metadata (seperti koordinat EPSG) dari file kecamatan pertama
        out_meta = src_files_to_mosaic[0].meta.copy()

        # Perbarui metadata untuk kanvas besar yang baru
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_trans,
            "nodata": 0
        })

        # Tentukan nama file output gabungan
        nama_file_gabungan = f"Peta_Karawang_Model_Skenario 9_Periode_{per}.tif"
        out_fp = os.path.join(OUTPUT_DIR, nama_file_gabungan)

        # Simpan file gabungan ke Google Drive & suntikkan kembali warnanya
        with rasterio.open(out_fp, "w", **out_meta) as dest:
            dest.write(mosaic)
            dest.write_colormap(1, warna_fase) # <--- Penting agar tetap berwarna!

        print(f"   ✅ Berhasil digabungkan: {nama_file_gabungan}")

        # Tutup semua file sumber untuk melegakan RAM Colab
        for src in src_files_to_mosaic:
            src.close()

    except Exception as e:
        print(f"   ❌ Terjadi kesalahan saat menggabungkan periode {per}: {e}")

print(f"\n🎉 SEMUA PROSES SELESAI!")
print(f"Silakan cek 6 Peta Kabupaten utuh Kakak di folder:\n📁 {OUTPUT_DIR}")

Google Drive is already mounted. Unmounting first...
Mounted at /content/drive
🌍 MEMULAI PENGGABUNGAN PETA (HANYA DARI FOLDER BARU)

⏳ Memproses Peta Periode 02...
   🔍 Ditemukan 30 kecamatan. Memulai penjahitan (merge)...
   ✅ Berhasil digabungkan: Peta_Karawang_Model_Skenario 9_Periode_02.tif

⏳ Memproses Peta Periode 04...
   🔍 Ditemukan 30 kecamatan. Memulai penjahitan (merge)...
   ✅ Berhasil digabungkan: Peta_Karawang_Model_Skenario 9_Periode_04.tif

⏳ Memproses Peta Periode 06...
   🔍 Ditemukan 30 kecamatan. Memulai penjahitan (merge)...
   ✅ Berhasil digabungkan: Peta_Karawang_Model_Skenario 9_Periode_06.tif

⏳ Memproses Peta Periode 08...
   🔍 Ditemukan 30 kecamatan. Memulai penjahitan (merge)...
   ✅ Berhasil digabungkan: Peta_Karawang_Model_Skenario 9_Periode_08.tif

⏳ Memproses Peta Periode 10...
   🔍 Ditemukan 30 kecamatan. Memulai penjahitan (merge)...
   ✅ Berhasil digabungkan: Peta_Karawang_Model_Skenario 9_Periode_10.tif

⏳ Memproses Peta Periode 12...
   🔍 Ditemukan 3